# Chicago Restaurant Inspection Failure Classification

## Load and prepare the data:


In [28]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

inspections = pd.read_csv("data/raw/Food_Inspections_20261012.csv")
licenses = pd.read_csv("data/raw/Business_Licenses_-_Current_Active_20261012.csv")
community = pd.read_csv("data/raw/CommunityDataSnapshots_Historical_455993172616713182.csv")

inspection_columns = [
    "License #", "Facility Type", "Inspection Date",
    "Inspection Type", "Risk", "Results", "Violations"
]
inspections = inspections[inspection_columns].copy()

licenses = licenses[["LICENSE NUMBER", "COMMUNITY AREA"]].copy()
community = community[["GEOID", "GEOG", "MEDINC"]].copy()

inspections["Inspection Date"] = pd.to_datetime(
    inspections["Inspection Date"], errors="coerce"
)

inspections["License #"] = pd.to_numeric(
    inspections["License #"], errors="coerce"
)
licenses["LICENSE NUMBER"] = pd.to_numeric(
    licenses["LICENSE NUMBER"], errors="coerce"
)
licenses["COMMUNITY AREA"] = pd.to_numeric(
    licenses["COMMUNITY AREA"], errors="coerce"
)
community["GEOID"] = pd.to_numeric(
    community["GEOID"], errors="coerce"
)

inspections = inspections[
    (inspections["Facility Type"] == "Restaurant")
    & inspections["Violations"].notna()
    & (inspections["Violations"] != "")
].drop_duplicates()

licenses = licenses.dropna().drop_duplicates()
licenses = licenses[licenses["LICENSE NUMBER"] > 0]

duplicate_licenses = licenses["LICENSE NUMBER"].duplicated(keep=False)
licenses = licenses[~duplicate_licenses]

community = community.dropna(subset=["GEOID"])
community = community.drop_duplicates(subset=["GEOID"])

data = inspections.merge(
    licenses,
    left_on="License #",
    right_on="LICENSE NUMBER",
    how="inner",
    validate="many_to_one"
)

data = data.merge(
    community,
    left_on="COMMUNITY AREA",
    right_on="GEOID",
    how="inner",
    validate="many_to_one"
)

data = data[data["Results"].isin(["Pass", "Fail"])].copy()

data["year"] = data["Inspection Date"].dt.year
data["month"] = data["Inspection Date"].dt.month

print(f"Eligible inspections: {len(data):,}")
display(data.head())

/tmp/ipykernel_25002/3522688514.py:14: DtypeWarning: Columns (9: ZIP CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  licenses = pd.read_csv("data/raw/Business_Licenses_-_Current_Active_20261012.csv")


Eligible inspections: 58,818


,License #,Facility Type,Inspection Date,Inspection Type,Risk,Results,Violations,LICENSE NUMBER,COMMUNITY AREA,GEOID,GEOG,MEDINC,year,month
0,2222369.0,Restaurant,2026-10-09,Canvass,Risk 1 (High),Pass,"55. PHYSICAL FACILITIES INSTALLED, MAINTAINED ...",2222369,38.0,38,Grand Boulevard,50075.107296,2026,10
1,2749668.0,Restaurant,2026-10-09,Canvass,Risk 1 (High),Fail,2. CITY OF CHICAGO FOOD SERVICE SANITATION CER...,2749668,24.0,24,West Town,141208.345621,2026,10
2,3002330.0,Restaurant,2026-10-09,Canvass,Risk 1 (High),Fail,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLI...,3002330,38.0,38,Grand Boulevard,50075.107296,2026,10
3,3046001.0,Restaurant,2026-10-08,Complaint,Risk 2 (Medium),Fail,"1. PERSON IN CHARGE PRESENT, DEMONSTRATES KNOW...",3046001,25.0,25,Austin,47532.573290,2026,10
4,2083875.0,Restaurant,2026-10-08,Canvass,Risk 1 (High),Fail,16. FOOD-CONTACT SURFACES: CLEANED & SANITIZED...,2083875,24.0,24,West Town,141208.345621,2026,10


## Feature:

In [35]:
cat = ['Risk', 'Inspection Type', 'GEOG']
num = ['year', 'month', 'MEDINC']

features = data[cat + num].copy()
target = data['Results'].eq('Fail').astype(int)

print('X shape:', features.shape)
display(
    target.value_counts()
    .sort_index()
    .rename(index={0: 'Pass (0)', 1: 'Fail (1)'})
    .to_frame('count')
)

X shape: (58818, 6)


,count
Results,
Pass (0),41460
Fail (1),17358


## Classification:

In [44]:
prep = ColumnTransformer([
    (
        'numeric',
        Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]),
        num
    ),
    (
        'categorical',
        Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]),
        cat
    )
])

def learn_classifier():
    return Pipeline([
        ('prep', prep),
        ('model', LogisticRegression(
            max_iter=1000,
            class_weight='balanced'
        ))
    ])

baseline = DummyClassifier(strategy='most_frequent')
classifier = learn_classifier()

print(baseline)
print(classifier)

DummyClassifier(strategy='most_frequent')
Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['year', 'month', 'MEDINC']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
         

## Evaluation:

In [45]:
def evaluate_classifier(model, X, y, cv, scoring):
    results = cross_validate(
        model, X, y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    return {
        metric: results[f'test_{metric}'].mean()
        for metric in scoring
    }


kf = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=1
)

scoring = {
    'accuracy': 'accuracy',
    'precision_fail': make_scorer(precision_score, zero_division=0),
    'recall_fail': make_scorer(recall_score, zero_division=0),
    'f1_fail': make_scorer(f1_score, zero_division=0)
}


baseline_scores = evaluate_classifier(baseline, features, target, kf, scoring)
classifier_scores = evaluate_classifier(classifier, features, target, kf, scoring)

scores = pd.DataFrame(
    [baseline_scores, classifier_scores],
    index=['Majority baseline', 'Logistic regression']
)

display(scores.style.format('{:.3f}'))

,accuracy,precision_fail,recall_fail,f1_fail
Majority baseline,0.705,0.000,0.000,0.000
Logistic regression,0.590,0.389,0.676,0.493


## Analysis:
Our team’s balanced logistic regression model achieved 59.0% accuracy, 38.9% failure precision, 67.6% failure recall, and a 49.3% F1-score.

Although our model had lower accuracy than the baseline, it identified about 67.6% of actual failed inspections. However, it also made many false predictions, since only 38.9% of inspections predicted to fail actually failed.

Our results show a trade-off between overall accuracy and detecting failed inspections. We believe our model could improve by using more useful features or trying other models.